# 01 - Download ComStock source data

Public research notebook for the STARS workflow. Dataset files and generated artifacts are intentionally excluded from Git. See `../data/README.md` before running.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
(DATA_DIR / "raw").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "interim").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT / "src"))


In [ ]:
import pandas as pd

# Replace 'path/to/file.parquet' with the actual path to your Parquet file
parquet_file_path = 'com-building/NY_G3600010_baseline_basic.parquet'

# Read the Parquet file into a pandas DataFrame
df = pd.read_parquet(parquet_file_path)

# Take a quick look at the data
print(df.head())  # Shows the first 5 rows
print(df.info())  # Displays column data types and other info

In [ ]:
df.shape

In [ ]:
df.shape

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# Create an S3 client configured for unsigned (public) access
s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))

bucket_name = "oedi-data-lake"
prefix = "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/comstock_amy2018_release_2/weather/amy2018/"

# Use a paginator to handle buckets with many objects
paginator = s3_client.get_paginator("list_objects_v2")

pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

for page in pages:
    if "Contents" in page:
        for obj in page["Contents"]:
            # obj["Key"] contains the full path (key) within the bucket
            print(obj["Key"])

In [ ]:
import os
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# S3 bucket and prefix
bucket_name = "oedi-data-lake"
prefix = (
    "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/"
    "2024/comstock_amy2018_release_2/weather/amy2018/"
)

# Create an S3 client that does not require AWS credentials
s3_client = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# Local directory where you want to download the data
local_root_dir = str(DATA_DIR / "raw" / "comstock")

# Create a paginator to handle large result sets
paginator = s3_client.get_paginator("list_objects_v2")
pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

# Loop through all pages
for page in pages:
    if "Contents" in page:
        # Loop through each object in the current page
        for obj in page["Contents"]:
            s3_key = obj["Key"]

            # Skip placeholder keys representing "folders" in S3
            if s3_key.endswith("/"):
                continue

            # Build the relative and local paths
            relative_path = s3_key[len(prefix):]
            local_path = os.path.join(local_root_dir, relative_path)

            # Ensure local directory structure exists
            local_dir = os.path.dirname(local_path)
            if not os.path.exists(local_dir):
                os.makedirs(local_dir, exist_ok=True)

            # Download the file from S3
            print(f"Downloading {s3_key} to {local_path}...")
            s3_client.download_file(bucket_name, s3_key, local_path)

print("All files have been downloaded.")
